# 🟪 RF-DETR — Drowsy Driver 6-class (Colab T4)

**Model non-YOLO (Transformer)** — RF-DETR của Roboflow, backbone **DINOv2**, SOTA (>60 AP COCO).

`Runtime → Change runtime type → T4 GPU → Run all`

## Khác biệt quan trọng so với YOLO
| | YOLO11 / YOLO26 | **RF-DETR** |
|---|---|---|
| Kiến trúc | CNN | **Transformer (DINOv2 + DETR decoder)** |
| Format dataset | `yolov11` / `yolo26` (txt) | **`coco`** (JSON) ← KHÁC! |
| Package | `ultralytics` | **`rfdetr`** |
| Hậu xử lý | NMS (YOLO11) / NMS-free (YOLO26) | **NMS-free** |
| Predict trả về | `results.boxes` | `supervision` Detections |

→ Bài báo cáo IS54A: so sánh **CNN (YOLO) vs Transformer (RF-DETR)** trên cùng dataset 6-class.

## Dataset 6 class
`close_eyeL`, `close_eyeR`, `open_eyeL`, `open_eyeR`, `yawn`, `no_yawn`

In [ ]:
# 1 — GPU + cài rfdetr (PHẢI có [train,loggers] để train được) + supervision
!nvidia-smi -L
%pip install -q -U "rfdetr[train,loggers]" supervision roboflow
import os, json, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import supervision as sv
print('✅  supervision', sv.__version__)
HOME = Path('/content')

In [ ]:
# 2 — Download dataset, FORMAT = 'coco'  (RF-DETR cần COCO, KHÔNG phải YOLO txt)
from roboflow import Roboflow

API_KEY = 'qI3lEKlNpIZpNENdk3MH'    # ← API key của bạn
FORMAT  = 'coco'                    # ← RF-DETR đọc COCO JSON
rf = Roboflow(api_key=API_KEY)

# ⚠️ Slug project Roboflow LUÔN viết thường! 'Datio_yolo' (hoa) → lỗi 404.
PROJECT_TRY = ['datio_yolo', 'driver-yawn', 'driver-yawn-wh6wj']
dataset = None
for proj in PROJECT_TRY:
    try:
        dataset = rf.workspace('nguyen-tuan-dat').project(proj).version(1).download(FORMAT)
        print(f'✅  Dùng project: {proj}  (format={FORMAT})')
        break
    except Exception:
        print(f'  ⏭️  {proj} không tải được, thử tiếp...')
if dataset is None:
    raise SystemExit('❌  Không project nào tải được — kiểm tra tên trên Roboflow')

DATASET_DIR = dataset.location
print('  DATASET_DIR =', DATASET_DIR)

In [ ]:
# 3 — Kiểm tra cấu trúc COCO (train/valid/test + _annotations.coco.json)
DD = Path(DATASET_DIR)
splits = [d.name for d in DD.iterdir() if d.is_dir()]
print('  Splits:', splits)

# Đọc classes từ COCO json của train
train_json = DD/'train'/'_annotations.coco.json'
with open(train_json) as f: coco = json.load(f)
cats = sorted(coco['categories'], key=lambda c: c['id'])
CLASSES = [c['name'] for c in cats]
print(f'  Categories ({len(cats)}):')
for c in cats: print(f'    id={c["id"]:<3} {c["name"]}')

for sp in ['train','valid','test']:
    jp = DD/sp/'_annotations.coco.json'
    if jp.exists():
        with open(jp) as f: cj = json.load(f)
        print(f'  {sp:<6}: {len(cj["images"])} ảnh, {len(cj["annotations"])} box')

# ⚠️ Roboflow COCO thường có category id=0 là supercategory (không phải class thật)
if cats and cats[0]['name'].lower() in ('drowsy','driver','none','workspace','objects'):
    print('  ⚠️  id=0 có thể là supercategory — RF-DETR tự xử lý, mAP per-class xem ở cuối')

---
## 🎯 Train RF-DETR

- **T4 (15GB)**: `batch_size=4`, `grad_accum_steps=4` → batch hiệu dụng 16
- Dataset nhỏ → `epochs=60`, `lr=1e-4`
- Scale: `RFDETRNano` (nhanh, hợp Android) → `RFDETRSmall` → `RFDETRMedium` → `RFDETRLarge` (mAP cao nhất)

In [ ]:
# 4 — TRAIN
# RF-DETR pretrain sẵn → hội tụ nhanh (epoch 1 đã ~0.58 mAP). 10 epoch là đủ cho đồ án (~50 phút).
from rfdetr import RFDETRSmall   # NHANH hơn: RFDETRNano | mAP cao hơn (chậm hơn): RFDETRMedium

OUTPUT_DIR = str(HOME/'rfdetr_out')
model = RFDETRSmall()

model.train(
    dataset_dir      = DATASET_DIR,
    epochs           = 10,       # đủ cho RF-DETR (hội tụ sớm). Tăng lên 20-30 nếu muốn mAP cao hơn
    batch_size       = 4,        # T4: giữ 4 (tăng dễ hết RAM)
    grad_accum_steps = 4,        # batch hiệu dụng = 16
    lr               = 1e-4,
    output_dir       = OUTPUT_DIR,
    tensorboard      = True,
    early_stopping   = True,     # tự dừng khi mAP plateau
)
print('✅  Train xong — checkpoint trong', OUTPUT_DIR)

In [ ]:
# 5 — Load checkpoint tốt nhất (EMA) + predict demo
from rfdetr import RFDETRSmall
import glob, cv2
from pathlib import Path

ckpts = [f'{OUTPUT_DIR}/checkpoint_best_ema.pth',
         f'{OUTPUT_DIR}/checkpoint_best_regular.pth',
         f'{OUTPUT_DIR}/checkpoint.pth']
BEST = next((c for c in ckpts if os.path.exists(c)), None)
print('  Dùng checkpoint:', BEST)
best_model = RFDETRSmall(pretrain_weights=BEST)

# Chọn split có ảnh để demo: test → valid → train (dataset có thể KHÔNG có test)
DD = Path(DATASET_DIR)
def pick_split():
    for sp in ['test','valid','train']:
        d = DD/sp
        if d.exists() and (list(d.glob('*.jpg'))+list(d.glob('*.png'))): return d
    return DD
EVAL_DIR = pick_split()
print('  Demo split:', EVAL_DIR.name)
imgs = sorted(list(EVAL_DIR.glob('*.jpg'))+list(EVAL_DIR.glob('*.png')))[:6]

box_ann = sv.BoxAnnotator(thickness=2)
lbl_ann = sv.LabelAnnotator(text_scale=0.4)
fig, ax = plt.subplots(2,3, figsize=(16,9))
for a, ip in zip(ax.flat, imgs):
    det = best_model.predict(str(ip), threshold=0.4)
    img = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
    labels = [f'{CLASSES[c] if c < len(CLASSES) else c} {p:.2f}'
              for c,p in zip(det.class_id, det.confidence)]
    img = box_ann.annotate(img, det)
    img = lbl_ann.annotate(img, det, labels)
    a.imshow(img); a.axis('off')
plt.suptitle('RF-DETR — predictions', fontweight='bold')
plt.tight_layout(); plt.savefig(HOME/'rfdetr_pred.png', dpi=120, bbox_inches='tight'); plt.show()

In [ ]:
# 6 — Đánh giá mAP (tự tìm split có annotation COCO: test → valid → train)
from supervision.metrics import MeanAveragePrecision
from pathlib import Path
DD = Path(DATASET_DIR)

def pick_coco():
    for sp in ['test','valid','train']:
        j = DD/sp/'_annotations.coco.json'
        if j.exists(): return DD/sp, j
    hits = list(DD.rglob('_annotations.coco.json'))
    return (hits[0].parent, hits[0]) if hits else (None, None)

eval_dir, eval_json = pick_coco()
print('  Eval split:', eval_dir)
assert eval_json, '❌ Không tìm thấy _annotations.coco.json nào'

ds_test = sv.DetectionDataset.from_coco(
    images_directory_path=str(eval_dir),
    annotations_path=str(eval_json))
print(f'  {len(ds_test)} ảnh — đang chạy inference...')

predictions, targets = [], []
for path, _img, ann in ds_test:
    det = best_model.predict(path, threshold=0.25)
    predictions.append(det); targets.append(ann)

try:
    result = MeanAveragePrecision().update(predictions, targets).compute()
    print(f'\n  ┌─ RF-DETR mAP ─────────────')
    print(f'  │ mAP50-95 = {result.map50_95*100:.2f}%')
    print(f'  │ mAP50    = {result.map50*100:.2f}%')
    print(f'  │ mAP75    = {result.map75*100:.2f}%')
    print(f'  └───────────────────────────')
    RF_MAP50, RF_MAP5095 = float(result.map50), float(result.map50_95)
except Exception as e:
    print('  ⚠️  mAP qua supervision lỗi:', str(e)[:100])
    print('  → Xem mAP trong log train (dòng "Best EMA mAP improved to ...")')
    RF_MAP50 = RF_MAP5095 = 0.0

In [ ]:
# 7 — Lưu Drive + export ONNX + download
from google.colab import drive, files
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/DrowsyDriver_Results'); OUT.mkdir(parents=True, exist_ok=True)

if BEST:
    shutil.copy(BEST, OUT/'rfdetr_small_best.pth')
    print('✅  Lưu Drive: rfdetr_small_best.pth')

summary = {'model':'rfdetr-small','format':'coco','classes':CLASSES,
           'map50':round(RF_MAP50,4),'map50_95':round(RF_MAP5095,4)}
(OUT/'summary_rfdetr.json').write_text(json.dumps(summary, indent=2))

# Export ONNX (để deploy / so sánh tốc độ)
try:
    best_model.export(output_dir=str(HOME/'rfdetr_onnx'))
    onnx = glob.glob(f'{HOME}/rfdetr_onnx/*.onnx')
    if onnx:
        shutil.copy(onnx[0], OUT/'rfdetr_small.onnx')
        print('✅  Export ONNX:', Path(onnx[0]).name)
except Exception as e:
    print('  ⏭️  ONNX export skip:', str(e)[:80])

if BEST: files.download(BEST)

---
## 📊 Đưa vào báo cáo: CNN vs Transformer

Chạy đủ 3 file rồi điền bảng này (mAP đọc từ output mỗi file):

| Họ | Model | Format | mAP50 | mAP50-95 | Tham số |
|----|-------|--------|-------|----------|---------|
| CNN | YOLO11s | yolov11 | ? | ? | 9.4M |
| CNN | YOLO26s | yolo26 | ? | ? | ~9M |
| **Transformer** | **RF-DETR-S** | **coco** | ? | ? | ~30M |

**Nhận xét mẫu cho báo cáo:**
- RF-DETR (transformer) thường **mAP50-95 cao hơn** YOLO nhờ DINOv2 backbone + attention toàn cục → định vị box chính xác hơn.
- YOLO **nhẹ & nhanh hơn** → hợp deploy Android. RF-DETR-Nano là lựa chọn cân bằng nếu muốn transformer trên mobile.
- Cả 2 đều **NMS-free** (YOLO26 & RF-DETR) → pipeline gọn.

> Muốn ensemble RF-DETR + YOLO? Cùng cách per-class WBF như `colab_combine_6class.ipynb`, chỉ cần wrap `best_model.predict()` trả về boxes normalized.